# Lezione 5. Predizione senza modello: Monte Carlo e Differenze Temporali

**Durata stimata:** 90 minuti
**Capitolo del manuale:** [06_prediccion_sin_modelo.md](../manual/06_prediccion_sin_modelo.md)

## Dalle equazioni note all'esperienza pura

Nella lezione precedente hai risolto il Processo Decisionale di Markov (PDM) con la Programmazione Dinamica (DP): un insieme di metodi potenti, che però richiede di conoscere in anticipo il modello completo dell'ambiente, cioè le probabilità di transizione e la funzione di ricompensa. In un problema reale —l'irrigazione di una coltura, la manutenzione di una flotta di pesca, un protocollo di cura— quasi mai si dispone di quel modello esatto.

Oggi cambi approccio: impari **senza modello** (*model-free*). Invece di calcolare il valore di ogni stato con equazioni note, lo stimi dall'esperienza, episodio dopo episodio, come imparerebbe chiunque un mestiere nuovo osservando. Programmerai le due famiglie di metodi che fanno questo: Monte Carlo (MC) e le Differenze Temporali (TD, dall'inglese *Temporal-Difference*). Entrambe risolvono lo stesso compito —la **predizione**: stimare Vπ(s) senza conoscere il modello— ma ci arrivano per strade diverse, con conseguenze pratiche che confronterai nel codice tra poco.

### Obiettivi di oggi

- Spiegare che cosa significa "predizione senza modello" e in cosa si differenzia dalla Programmazione Dinamica
- Programmare Monte Carlo a prima visita per stimare Vπ a partire da episodi completi
- Programmare TD(0) e capire il bootstrapping: aggiornare una stima con un'altra stima
- Confrontare la velocità di convergenza dei due metodi e metterla in relazione con il compromesso tra bias e varianza

> 🎙️ **Traccia della lezione:** apri chiedendo "se nessuno vi desse la mappa del traffico, come stimereste quanto dura un tragitto?". Lascia che rispondano "facendolo più volte" prima di nominare Monte Carlo: è l'intuizione che formalizzerai oggi.


## 1. Che cosa significa "predizione" senza modello

Ricorda che Vπ(s) è il ritorno atteso se segui la politica π a partire dallo stato s. Con la DP calcolavi questo valore risolvendo equazioni sul modello. Senza modello, l'unica fonte di informazione che ti resta è l'esperienza reale: lasci che l'agente agisca secondo π, osservi le ricompense che riceve e, con quei dati, stimi Vπ(s).

È la differenza tra calcolare la durata media di un tragitto con una mappa del traffico perfetta e stimarla cronometrando il tragitto molte volte.


## 2. Monte Carlo: impara dall'episodio completo

MC è il più diretto dei due metodi: lascia che l'episodio termini, calcola il ritorno totale ottenuto a partire da ogni stato visitato e usa quel ritorno come campione del valore reale. Ripeti questo con molti episodi e fai la media. Per la legge dei grandi numeri, quella media converge al vero Vπ(s).

Pensa al tragitto da casa al lavoro: l'approccio MC aspetta che tu arrivi, annota il tempo totale e aggiorna la stima con quel dato completo. Più giorni ripeti il tragitto, migliore sarà la stima, ma devi arrivare ogni volta per imparare qualcosa: se l'episodio non termina, MC non ha nulla da mediare.

All'interno di MC esistono due varianti, a seconda di quante volte conti ogni stato all'interno dello stesso episodio:

- **Prima visita** (*first-visit*): usi solo il ritorno dalla prima volta in cui compare uno stato nell'episodio, anche se si ripete più avanti
- **Ogni visita** (*every-visit*): usi il ritorno da ogni occorrenza dello stato, non solo la prima

Entrambe le varianti convergono allo stesso valore con un numero sufficiente di episodi; oggi programmi la prima.

> 🎙️ **Traccia della lezione:** chiedi "cosa succederebbe se uno stato comparisse tre volte nello stesso episodio?" prima di spiegare la differenza tra prima visita e ogni visita. Lascia che intuiscano che "prima visita" è l'opzione più semplice da programmare.


## 3. Differenze Temporali: indovina a partire dalla tua stessa previsione

TD(0) risolve lo stesso problema con un'idea diversa: non aspettare la fine dell'episodio, aggiorna la tua stima dopo ogni passo. Per farlo combina la ricompensa appena osservata con la tua stima attuale del valore dello stato successivo, invece di aspettare il ritorno reale completo. Questa tecnica si chiama **bootstrapping**: usi una stima per correggere un'altra stima.

Seguendo l'analogia del tragitto: invece di aspettare di arrivare per giudicare se il viaggio sta andando bene, correggi la tua previsione a ogni tratto. Quando esci sotto la pioggia, rivedi al rialzo la stima. Quando ti immetti su un'autostrada senza code, la rivedi al ribasso. Non serve arrivare a casa per imparare qualcosa di utile a ogni tratto.

La regola di aggiornamento di TD(0) è questa:

```
V(S) ← V(S) + α · [R + γ·V(S') − V(S)]
```

Dove α è il tasso di apprendimento, R è la ricompensa osservata nel fare il passo, γ è il fattore di sconto e V(S') è la tua stima attuale dello stato successivo (zero se S' è terminale). Il termine tra parentesi quadre è l'**errore TD**: la differenza tra ciò che hai appena osservato più ciò che credi valga il resto del percorso, e ciò che credevi prima di fare il passo.

Poiché TD non aspetta la fine, funziona anche con episodi incompleti o compiti che non terminano mai, cosa che MC non può fare.

> 🎙️ **Traccia della lezione:** scrivi la formula alla lavagna e indica ogni simbolo mentre la leggi ad alta voce: "il nuovo valore è il vecchio valore, più un po' dell'errore tra ciò che ho appena visto e ciò che già credevo". È la frase che la classe ripeterà tra poco, già nel codice.


## 4. Bias e varianza: il grande compromesso

Questa differenza di meccanismo ha una conseguenza statistica diretta:

- MC fa la media di ritorni completi e reali, quindi la sua stima è **non distorta** (*unbiased*): in media centra il valore vero. Il prezzo è una **varianza alta**, perché ogni ritorno dipende dall'intera sequenza di azioni e ricompense casuali fino alla fine dell'episodio
- TD aggiorna con una sola ricompensa osservata più una stima ancora imperfetta del resto, quindi è **distorto** (*biased*) finché quella stima non converge. In cambio ha una **varianza bassa**, perché ogni aggiornamento dipende solo da un passo

Con infiniti episodi e un tasso di apprendimento che decresce in modo adeguato, entrambi i metodi convergono allo stesso Vπ(s). Con dati finiti —la situazione reale quasi sempre— possono dare risposte diverse, e di solito conviene accettare un po' di distorsione in cambio di molta meno varianza. Per questo TD è così usato nella pratica: impara in fretta e con stime stabili, anche se impiega un po' a "liberarsi" della distorsione iniziale.


## 5. Manos a la obra: la passeggiata aleatoria

Confronterai i due metodi nella **passeggiata aleatoria** (*random walk*), un ambiente minimo ma molto illustrativo: una catena di cinque stati non terminali, con uno stato terminale a ciascuna estremità. Dallo stato centrale, a ogni passo ti muovi a sinistra o a destra con la stessa probabilità. Se arrivi all'estremità destra ricevi una ricompensa di uno; se arrivi all'estremità sinistra, ricevi zero.

È un ambiente così semplice che puoi calcolare a mano il valore reale di ogni stato, il che ti permette di misurare con precisione quanto velocemente converge ciascun metodo. Costruisci il codice cella per cella, eseguendo ciascuna prima di passare alla successiva.

> 🎙️ **Traccia della lezione:** disegna alla lavagna la catena di sette stati (due terminali, cinque interni) e segna lo stato iniziale al centro. È l'immagine che la classe avrà presente per il resto della sessione.


In [ ]:
# Solo necesitas numpy y matplotlib: ambos ya vienen preinstalados en Google Colab
import numpy as np
import matplotlib.pyplot as plt


### L'ambiente: definizione

Numera gli stati da zero a sei: 0 e 6 sono terminali (sinistra e destra); da uno a cinque sono gli stati interni A, B, C, D, E, e l'episodio parte sempre dallo stato centrale, C.

Poiché l'ambiente è così semplice, il valore reale di ogni stato interno ha una formula chiusa: V(stato i) = i / 6. La userai più sotto per misurare l'errore delle tue stime.


In [ ]:
# 🎙️ Guion: pide a la clase que calcule a mano V(C), el estado central,
# con la fórmula i / 6 antes de ejecutar la celda (la respuesta es 0.5).
N_ESTADOS_INTERNOS = 5
ESTADO_INICIAL = 3          # C, el estado central de la cadena
TERMINAL_IZQUIERDA = 0
TERMINAL_DERECHA = N_ESTADOS_INTERNOS + 1

# Valor real de cada estado interno: V(estado i) = i / (N_ESTADOS_INTERNOS + 1)
VALORES_REALES = np.array(
    [i / (N_ESTADOS_INTERNOS + 1) for i in range(1, N_ESTADOS_INTERNOS + 1)]
)
print("Valores reales de A, B, C, D, E:", VALORES_REALES)


### Simula un episodio

`generar_episodio` lascia che l'agente cammini dallo stato iniziale fino a cadere a un'estremità, e restituisce la lista di coppie (stato, ricompensa) che ha attraversato. Solo l'estremità destra dà ricompensa.


In [ ]:
def generar_episodio():
    '''
    Simula un episodio de paseo aleatorio y devuelve la lista de
    (estado, recompensa) que atraviesa, terminando en el estado terminal.
    '''
    estado = ESTADO_INICIAL
    trayectoria = []
    while True:
        # Con probabilidad 0.5 te mueves a la izquierda o a la derecha
        paso = -1 if np.random.rand() < 0.5 else 1
        siguiente_estado = estado + paso
        # Solo el extremo derecho da recompensa; el resto de pasos dan cero
        recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0
        trayectoria.append((estado, recompensa))
        estado = siguiente_estado
        if estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            break
    return trayectoria


### Verifica rapida

Genera un episodio di esempio e osserva la sequenza di stati che percorre. Poiché la passeggiata è casuale, ogni esecuzione dà un risultato diverso.

> 🎙️ **Traccia della lezione:** esegui la cella due o tre volte di seguito perché la classe veda che la lunghezza dell'episodio cambia ogni volta: alcuni percorsi sono brevi, altri fanno molti giri prima di terminare.


In [ ]:
np.random.seed(0)
episodio_ejemplo = generar_episodio()
print("Episodio de ejemplo (estado, recompensa):")
print(episodio_ejemplo)


### Monte Carlo a prima visita: la funzione

`monte_carlo_prediccion` percorre ogni episodio dalla fine all'inizio, accumula il ritorno G e, per ogni stato, si tiene il ritorno calcolato alla sua prima comparsa. Poi aggiorna V(s) verso quel ritorno con la regola incrementale V(s) ← V(s) + α·(G − V(s)).


In [ ]:
def monte_carlo_prediccion(n_episodios, alpha, gamma=1.0):
    '''
    Estima V para los estados internos 1..5 usando MC de primera visita
    con actualización incremental V(s) <- V(s) + alpha * (G - V(s)).
    '''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)  # inicialización neutra
    historial_rmse = []

    for _ in range(n_episodios):
        trayectoria = generar_episodio()

        # Calcula el retorno G_t desde cada paso hacia el final del episodio
        G = 0.0
        retornos_por_estado = {}
        for estado, recompensa in reversed(trayectoria):
            G = recompensa + gamma * G
            # Primera visita: si el estado se repite, nos quedamos con el
            # retorno calculado en la ocurrencia más temprana (la última
            # que procesamos al recorrer la trayectoria al revés)
            retornos_por_estado[estado] = G

        for estado, G in retornos_por_estado.items():
            if estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                idx = estado - 1  # los estados internos van de 1 a 5
                V[idx] += alpha * (G - V[idx])

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


### Provalo: esegui Monte Carlo

Esegui MC per 150 episodi con α = 0.1 e osserva quanto si avvicina V ai valori reali.


In [ ]:
np.random.seed(0)
V_mc, rmse_mc = monte_carlo_prediccion(n_episodios=150, alpha=0.1)

print("V estimado con MC:", V_mc)
print("Valores reales:   ", VALORES_REALES)
print(f"RMSE final: {rmse_mc[-1]:.4f}")


### TD(0): la funzione

`td0_prediccion` non aspetta la fine dell'episodio: a ogni passo calcola l'errore TD confrontando la ricompensa osservata più la tua stima dello stato successivo con la tua stima attuale, e aggiorna V(s) all'istante.


In [ ]:
def td0_prediccion(n_episodios, alpha, gamma=1.0):
    '''
    Estima V para los estados internos 1..5 con TD(0): actualiza en
    cada paso, sin esperar a que el episodio termine.
    '''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        estado = ESTADO_INICIAL
        while estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            paso = -1 if np.random.rand() < 0.5 else 1
            siguiente_estado = estado + paso
            recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0

            # Valor del siguiente estado: cero si es terminal, si no, la
            # estimación actual (esto es el "bootstrapping")
            if siguiente_estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                v_siguiente = 0.0
            else:
                v_siguiente = V[siguiente_estado - 1]

            idx = estado - 1
            error_td = recompensa + gamma * v_siguiente - V[idx]
            V[idx] += alpha * error_td

            estado = siguiente_estado

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


### Provalo: esegui TD(0)

Esegui anche TD(0) per 150 episodi con α = 0.1, e confronta il suo V finale con quello di MC.


In [ ]:
np.random.seed(0)
V_td, rmse_td = td0_prediccion(n_episodios=150, alpha=0.1)

print("V estimado con TD(0):", V_td)
print("Valores reales:      ", VALORES_REALES)
print(f"RMSE final: {rmse_td[-1]:.4f}")


## 6. Confronto: velocità di convergenza

Esegui i due metodi nelle stesse condizioni e disegna l'RMSE (la radice dell'errore quadratico medio, dall'inglese *root mean squared error*) rispetto al valore reale, episodio per episodio.


In [ ]:
np.random.seed(0)
N_EPISODIOS = 150
ALPHA = 0.1

_, rmse_mc = monte_carlo_prediccion(N_EPISODIOS, ALPHA)
_, rmse_td = td0_prediccion(N_EPISODIOS, ALPHA)

plt.figure(figsize=(8, 5))
plt.plot(rmse_mc, label="Monte Carlo (primera visita)")
plt.plot(rmse_td, label="TD(0)")
plt.xlabel("Episodios")
plt.ylabel("RMSE frente al valor real")
plt.title("Convergencia de MC vs TD(0) en el paseo aleatorio (α = 0.1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Come leggere il grafico

Vedrai tipicamente che TD(0) riduce l'errore più rapidamente di MC durante i primi episodi, grazie alla sua varianza più bassa, anche se entrambi finiscono per avvicinarsi a una zona di errore simile man mano che gli episodi aumentano. È la teoria di bias e varianza che hai visto prima, resa in forma di grafico.

> 🎙️ **Traccia della lezione:** prima di eseguire la cella precedente, chiedi "quale curva pensate che scenderà più in fretta all'inizio, e perché?". Conferma l'intuizione con il risultato.


## 7. Esperimento dal vivo: cambia il tasso di apprendimento

Prova tre valori di α (0.01, 0.05 e 0.15) con entrambi i metodi e osserva l'effetto. Un α troppo alto accelera l'apprendimento iniziale ma lascia più rumore residuo; un α troppo basso converge in modo più regolare, ma molto più lentamente.

> 🎙️ **Traccia della lezione:** distribuisci i tre valori di α tra gruppi diversi dell'aula e chiedi a ciascun gruppo di descrivere in una frase ciò che vede nel "proprio" grafico, prima di commentare insieme i tre.


In [ ]:
np.random.seed(0)
N_EPISODIOS_EXPERIMENTO = 150
ALPHAS_A_PROBAR = [0.01, 0.05, 0.15]

figura, ejes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for eje, alpha in zip(ejes, ALPHAS_A_PROBAR):
    _, rmse_mc_alpha = monte_carlo_prediccion(N_EPISODIOS_EXPERIMENTO, alpha)
    _, rmse_td_alpha = td0_prediccion(N_EPISODIOS_EXPERIMENTO, alpha)
    eje.plot(rmse_mc_alpha, label="MC")
    eje.plot(rmse_td_alpha, label="TD(0)")
    eje.set_title(f"α = {alpha}")
    eje.set_xlabel("Episodios")
    eje.grid(alpha=0.3)

ejes[0].set_ylabel("RMSE frente al valor real")
ejes[0].legend()
plt.tight_layout()
plt.show()


### Più episodi, lo stesso errore minimo?

Con α fisso, nessuno dei due metodi raggiunge esattamente errore zero: l'aggiornamento con un tasso di apprendimento costante lascia sempre un po' di rumore residuo, tanto più alto quanto è maggiore α. Per verificarlo con chiarezza, calcola la media di più esecuzioni con 1.000 episodi e confronta l'errore finale di MC e TD(0) per tre valori di α.

> 🎙️ **Traccia della lezione:** prima di eseguire, chiedi "se lascio girare questo per mille episodi invece di centocinquanta, l'errore arriverà a zero?". È una trappola comune: con α costante, non succede.


In [ ]:
def rmse_promedio(funcion_prediccion, n_episodios, alpha, n_ejecuciones=20):
    '''
    Repite el experimento n_ejecuciones veces con semillas distintas y
    promedia el historial de RMSE, para suavizar el ruido de una sola
    ejecución antes de comparar tasas de aprendizaje.
    '''
    acumulado = np.zeros(n_episodios)
    for ejecucion in range(n_ejecuciones):
        np.random.seed(ejecucion)
        _, historial = funcion_prediccion(n_episodios, alpha)
        acumulado += np.array(historial)
    return acumulado / n_ejecuciones


N_EPISODIOS_LARGO = 1000

print(f"{'α':>6} | {'RMSE final MC':>14} | {'RMSE final TD(0)':>17}")
for alpha in (0.01, 0.05, 0.1):
    rmse_mc_prom = rmse_promedio(monte_carlo_prediccion, N_EPISODIOS_LARGO, alpha)
    rmse_td_prom = rmse_promedio(td0_prediccion, N_EPISODIOS_LARGO, alpha)
    print(f"{alpha:>6} | {rmse_mc_prom[-1]:>14.4f} | {rmse_td_prom[-1]:>17.4f}")


## Punti chiave

- La predizione senza modello significa stimare Vπ(s) solo a partire dall'esperienza, senza conoscere le probabilità di transizione né la funzione di ricompensa
- MC fa la media dei ritorni completi degli episodi; non usa il bootstrapping e richiede che l'episodio termini
- La prima visita usa solo la prima occorrenza di uno stato per episodio; ogni visita le usa tutte
- TD(0) aggiorna dopo ogni passo con la ricompensa osservata più la tua stima dello stato successivo: questo è il bootstrapping
- MC è non distorto ma ha varianza alta; TD è distorto all'inizio ma ha varianza bassa, e in entrambi i casi converge con esperienza sufficiente
- Con un tasso di apprendimento α costante, nessun metodo raggiunge errore zero: resta un rumore residuo che cresce con α

## Chiusura e prossimi passi

Ora sai stimare il valore di una politica senza conoscere il modello. Nella Lezione 6 fai il passo successivo: usare la stessa idea delle Differenze Temporali per migliorare la politica, non solo valutarla, con due algoritmi di controllo classici —SARSA e Q-learning— sull'ambiente di Cliff Walking.

**Prima della prossima lezione:**
- **Apri** il notebook di pratica `clase05_practica_prediccion_sin_modelo.ipynb` e completa i due esercizi
- **Sperimenta** con altri valori di α e di numero di episodi, e annota in quale punto esatto si incrociano le curve di MC e TD(0)
- **Ripassa** il capitolo 6 del manuale se vuoi approfondire la derivazione matematica

> 🎙️ **Traccia della lezione:** chiudi ricordando la frase chiave di oggi: "TD aggiusta con ciò che è appena successo più ciò che già credeva; MC aspetta che il viaggio finisca per intero". È la distinzione che la prossima lezione riutilizza per passare dal valutare le politiche al migliorarle.
